[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

/home/user/miniconda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [16]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    # zero_grad, loop (forward, scale loss, backward), step
    model.train()
    total_loss = 0
    batch_cnt = len(micro_batches)
    optimizer.zero_grad()
    for x, y in micro_batches:
        out = model(x)
        loss = loss_fn(out, y) / batch_cnt
        loss.backward()
        total_loss += loss.item()
    optimizer.step()
    return total_loss

In [17]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 1.4106682538986206


In [18]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (7.4ms)
  ✅ [2/3] Returns loss value (1.7ms)
  ✅ [3/3] Parameters actually update (1.3ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (10.4ms total)
  Progress saved. Run status() to see your dashboard.



In [19]:
from torch_judge import hint
hint("gradient_accumulation")


💡 Hint for Gradient Accumulation:
   Zero grads once. For each micro-batch: forward, loss/n_batches, backward. Then optimizer.step(). The loss scaling ensures accumulated grads match a single large batch.

